# Chapter 9

## Section 9.1 - Model optimization techniques

In [13]:
# Section 9.1: Model-Level Optimization Techniques

import torch
import torch.nn as nn
from torch.quantization import quantize_dynamic, prepare, convert
from transformers import AutoModel, AutoTokenizer
import numpy as np
from typing import Dict, List, Optional, Tuple
import time
import asyncio
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache

# Set up quantization backend with fallback
def setup_quantization_backend():
    """Set up the best available quantization backend."""
    available_engines = torch.backends.quantized.supported_engines
    print(f"Available quantization engines: {available_engines}")
    
    # Try engines in order of preference
    preferred_engines = ['fbgemm', 'qnnpack', 'onednn']
    
    for engine in preferred_engines:
        if engine in available_engines:
            try:
                torch.backends.quantized.engine = engine
                print(f"Successfully set quantization engine to: {engine}")
                return True
            except RuntimeError as e:
                print(f"Failed to set engine {engine}: {e}")
                continue
    
    print("No quantization engines available - quantization will be disabled")
    return False

# Initialize quantization backend
QUANTIZATION_AVAILABLE = setup_quantization_backend()
print(f"Quantization available: {QUANTIZATION_AVAILABLE}")

Available quantization engines: ['qnnpack', 'none']
Successfully set quantization engine to: qnnpack
Quantization available: True


In [14]:
# ==========================================
# Model Quantization Implementation
# ==========================================

class ModelQuantizer:
    """Implements various quantization techniques for model optimization."""
    
    def __init__(self, model: nn.Module):
        self.model = model
        self.quantized_model = None
        
    def dynamic_quantization(self, dtype=torch.qint8):
        """Apply dynamic quantization to reduce model size and increase speed."""
        if not QUANTIZATION_AVAILABLE:
            print("Quantization not available - returning original model")
            self.quantized_model = self.model
            return self.quantized_model
            
        try:
            self.quantized_model = quantize_dynamic(
                self.model, 
                {nn.Linear, nn.LSTM, nn.GRU}, 
                dtype=dtype
            )
            print("Dynamic quantization completed successfully")
            return self.quantized_model
        except Exception as e:
            print(f"Dynamic quantization failed: {e}")
            print("Returning original model as fallback")
            self.quantized_model = self.model
            return self.quantized_model
    
    def static_quantization(self, calibration_data):
        """Apply static quantization with calibration data."""
        if not QUANTIZATION_AVAILABLE:
            print("Quantization not available - using dynamic quantization fallback")
            return self.dynamic_quantization()
            
        try:
            # Create a copy for quantization
            model_copy = torch.nn.Sequential(*[layer for layer in self.model])
            model_copy.eval()
            
            # Get the current quantization engine
            current_engine = torch.backends.quantized.engine
            
            # Set quantization configuration
            model_copy.qconfig = torch.quantization.get_default_qconfig(current_engine)
            
            # Prepare model for quantization
            prepared_model = prepare(model_copy, inplace=False)
            
            # Calibrate with sample data
            with torch.no_grad():
                for data in calibration_data:
                    if isinstance(data, (list, tuple)):
                        prepared_model(*data)
                    else:
                        prepared_model(data)
            
            # Convert to quantized model
            self.quantized_model = convert(prepared_model, inplace=False)
            print("Static quantization completed successfully")
            return self.quantized_model
            
        except Exception as e:
            print(f"Static quantization failed: {e}. Using dynamic quantization fallback.")
            return self.dynamic_quantization()
    
    def compare_performance(self, test_input, iterations=100):
        """Compare performance between original and quantized models."""
        original_times = []
        quantized_times = []
        
        # Test original model
        self.model.eval()
        with torch.no_grad():
            for _ in range(iterations):
                start = time.time()
                try:
                    _ = self.model(test_input)
                    original_times.append(time.time() - start)
                except Exception as e:
                    print(f"Original model execution failed: {e}")
                    break
        
        # Test quantized model
        if self.quantized_model and self.quantized_model is not self.model:
            self.quantized_model.eval()
            with torch.no_grad():
                for _ in range(iterations):
                    start = time.time()
                    try:
                        _ = self.quantized_model(test_input)
                        quantized_times.append(time.time() - start)
                    except Exception as e:
                        print(f"Quantized model execution failed: {e}")
                        break
        else:
            print("No quantized model available for comparison")
            quantized_times = original_times.copy()
        
        if not original_times or not quantized_times:
            return {
                'original_avg_time': 0,
                'quantized_avg_time': 0,
                'speedup': 1.0,
                'size_reduction': 0
            }
        
        return {
            'original_avg_time': np.mean(original_times),
            'quantized_avg_time': np.mean(quantized_times),
            'speedup': np.mean(original_times) / np.mean(quantized_times) if quantized_times else 1.0,
            'size_reduction': self._calculate_size_reduction()
        }
    
    def _calculate_size_reduction(self):
        """Calculate model size reduction after quantization."""
        if not self.quantized_model:
            return 0
        
        original_size = sum(p.numel() * p.element_size() for p in self.model.parameters())
        quantized_size = sum(p.numel() * p.element_size() for p in self.quantized_model.parameters())
        
        return 1 - (quantized_size / original_size)

In [15]:
# ==========================================
# Model Distillation Implementation
# ==========================================

class ModelDistiller:
    """Knowledge distillation for creating smaller, specialized models."""
    
    def __init__(self, teacher_model: nn.Module, student_model: nn.Module, temperature: float = 4.0):
        self.teacher_model = teacher_model
        self.student_model = student_model
        self.temperature = temperature
        self.criterion = nn.KLDivLoss(reduction='batchmean')
        
    def distillation_loss(self, student_logits, teacher_logits, true_labels, alpha=0.7):
        """Calculate distillation loss combining soft and hard targets."""
        # Soft targets from teacher
        soft_teacher = torch.softmax(teacher_logits / self.temperature, dim=1)
        soft_student = torch.log_softmax(student_logits / self.temperature, dim=1)
        soft_loss = self.criterion(soft_student, soft_teacher) * (self.temperature ** 2)
        
        # Hard targets
        hard_loss = nn.CrossEntropyLoss()(student_logits, true_labels)
        
        return alpha * soft_loss + (1 - alpha) * hard_loss
    
    def train_student(self, dataloader, optimizer, epochs=10):
        """Train student model using knowledge distillation."""
        self.teacher_model.eval()
        self.student_model.train()
        
        for epoch in range(epochs):
            total_loss = 0
            for batch_idx, (data, targets) in enumerate(dataloader):
                optimizer.zero_grad()
                
                # Get teacher predictions (no gradients)
                with torch.no_grad():
                    teacher_logits = self.teacher_model(data)
                
                # Get student predictions
                student_logits = self.student_model(data)
                
                # Calculate distillation loss
                loss = self.distillation_loss(student_logits, teacher_logits, targets)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            print(f"Epoch {epoch+1}, Average Loss: {total_loss/len(dataloader):.4f}")


In [16]:
# ==========================================
# Inference Optimization
# ==========================================

class InferenceOptimizer:
    """Optimizes model inference for speed and throughput."""
    
    def __init__(self, model, tokenizer=None, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.model = model.to(device)
        self.tokenizer = tokenizer
        self.device = device
        self.cache = {}
        
    @lru_cache(maxsize=1000)
    def cached_encode(self, text: str):
        """Cache tokenization results for repeated inputs."""
        if self.tokenizer:
            return self.tokenizer(text, return_tensors='pt', padding=True, truncation=True)
        return text
    
    def optimize_model_for_inference(self):
        """Apply various optimizations for inference."""
        self.model.eval()
        
        # Disable gradient computation
        for param in self.model.parameters():
            param.requires_grad = False
        
        # Use TorchScript if possible
        try:
            self.model = torch.jit.script(self.model)
            print("Successfully converted to TorchScript")
        except Exception as e:
            print(f"TorchScript conversion failed: {e}")
        
        # Enable inference optimizations
        if hasattr(torch.backends, 'cudnn'):
            torch.backends.cudnn.benchmark = True
    
    def batch_inference(self, inputs: List[str], batch_size: int = 32):
        """Process multiple inputs in batches for better throughput."""
        results = []
        
        for i in range(0, len(inputs), batch_size):
            batch = inputs[i:i + batch_size]
            
            # Tokenize batch
            if self.tokenizer:
                encoded = self.tokenizer(batch, return_tensors='pt', padding=True, truncation=True)
                encoded = {k: v.to(self.device) for k, v in encoded.items()}
            else:
                encoded = batch
            
            # Inference
            with torch.no_grad():
                outputs = self.model(**encoded)
            
            results.extend(outputs.logits.cpu().numpy())
        
        return results
    
    async def async_inference(self, inputs: List[str], max_workers: int = 4):
        """Asynchronous inference for concurrent processing."""
        loop = asyncio.get_event_loop()
        
        def process_batch(batch):
            return self.batch_inference(batch)
        
        # Split inputs into chunks for parallel processing
        chunk_size = len(inputs) // max_workers
        chunks = [inputs[i:i + chunk_size] for i in range(0, len(inputs), chunk_size)]
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            tasks = [loop.run_in_executor(executor, process_batch, chunk) for chunk in chunks]
            results = await asyncio.gather(*tasks)
        
        # Flatten results
        return [item for sublist in results for item in sublist]

In [17]:
# ==========================================
# Hardware Acceleration Manager
# ==========================================

class HardwareAccelerationManager:
    """Manages hardware acceleration options for optimal performance."""
    
    def __init__(self):
        self.device_info = self._detect_hardware()
    
    def _detect_hardware(self) -> Dict[str, bool]:
        """Detect available hardware acceleration options."""
        return {
            'cuda': torch.cuda.is_available(),
            'cuda_device_count': torch.cuda.device_count() if torch.cuda.is_available() else 0,
            'mps': hasattr(torch.backends, 'mps') and torch.backends.mps.is_available(),
            'cpu_count': torch.get_num_threads()
        }
    
    def optimize_for_hardware(self, model: nn.Module) -> Tuple[nn.Module, str]:
        """Optimize model placement and configuration for available hardware."""
        if self.device_info['cuda']:
            device = 'cuda'
            model = model.cuda()
            
            # Enable mixed precision if available
            if torch.cuda.is_available():
                torch.backends.cudnn.benchmark = True
                
        elif self.device_info['mps']:
            device = 'mps'
            model = model.to('mps')
            
        else:
            device = 'cpu'
            # Optimize for CPU
            torch.set_num_threads(self.device_info['cpu_count'])
            
        return model, device
    
    def setup_distributed_training(self, model: nn.Module, world_size: int):
        """Setup model for distributed training across multiple GPUs."""
        if self.device_info['cuda_device_count'] > 1:
            model = nn.DataParallel(model)
            print(f"Using {self.device_info['cuda_device_count']} GPUs")
        
        return model


In [18]:
# ==========================================
# Caching and Batching System
# ==========================================

class IntelligentCache:
    """Advanced caching system with TTL and size limits."""
    
    def __init__(self, max_size: int = 10000, ttl: int = 3600):
        self.cache = {}
        self.access_times = {}
        self.max_size = max_size
        self.ttl = ttl
    
    def get(self, key: str) -> Optional[any]:
        """Get item from cache if valid."""
        if key in self.cache:
            # Check TTL
            if time.time() - self.access_times[key] < self.ttl:
                self.access_times[key] = time.time()
                return self.cache[key]
            else:
                # Expired
                del self.cache[key]
                del self.access_times[key]
        return None
    
    def put(self, key: str, value: any):
        """Add item to cache with LRU eviction."""
        # Evict if at capacity
        if len(self.cache) >= self.max_size:
            # Remove least recently used
            oldest_key = min(self.access_times.keys(), key=lambda k: self.access_times[k])
            del self.cache[oldest_key]
            del self.access_times[oldest_key]
        
        self.cache[key] = value
        self.access_times[key] = time.time()

class BatchProcessor:
    """Intelligent batching system for improved throughput."""
    
    def __init__(self, model_func, batch_size: int = 32, max_wait_time: float = 0.1):
        self.model_func = model_func
        self.batch_size = batch_size
        self.max_wait_time = max_wait_time
        self.pending_requests = []
        self.results = {}
        
    async def add_request(self, request_id: str, data: any) -> any:
        """Add request to batch queue and wait for result."""
        self.pending_requests.append((request_id, data))
        
        # Trigger batch processing if batch is full or timeout
        if len(self.pending_requests) >= self.batch_size:
            await self._process_batch()
        else:
            # Wait for timeout or more requests
            await asyncio.sleep(self.max_wait_time)
            if self.pending_requests:  # Still have pending requests
                await self._process_batch()
        
        # Return result
        return self.results.pop(request_id, None)
    
    async def _process_batch(self):
        """Process current batch of requests."""
        if not self.pending_requests:
            return
        
        batch_data = [data for _, data in self.pending_requests]
        request_ids = [req_id for req_id, _ in self.pending_requests]
        
        # Process batch
        batch_results = await self.model_func(batch_data)
        
        # Store results
        for req_id, result in zip(request_ids, batch_results):
            self.results[req_id] = result
        
        # Clear pending requests
        self.pending_requests.clear()


In [47]:
# ==========================================
# Example Usage and Performance Testing
# ==========================================

def example_usage():
    """Demonstrate model-level optimization techniques."""
    
    print("=== Model-Level Optimization Demo ===")
    print(f"PyTorch version: {torch.__version__}")
    
    # Check quantization support
    print(f"Quantization available: {QUANTIZATION_AVAILABLE}")
    if QUANTIZATION_AVAILABLE:
        print(f"Current quantization engine: {torch.backends.quantized.engine}")
    
    # Example models - Teacher (large) and Student (small)
    teacher_model = nn.Sequential(
        nn.Linear(768, 1024),
        nn.ReLU(),
        nn.Linear(1024, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 10)
    )
    
    student_model = nn.Sequential(
        nn.Linear(768, 256),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )
    
    teacher_params = sum(p.numel() for p in teacher_model.parameters())
    student_params = sum(p.numel() for p in student_model.parameters())
    
    print(f"Teacher model: {teacher_params:,} parameters")
    print(f"Student model: {student_params:,} parameters")
    print(f"Size reduction: {((teacher_params - student_params) / teacher_params) * 100:.1f}%")
    
    # 1. Knowledge Distillation
    print("\n1. Testing knowledge distillation...")
    try:
        # Create realistic synthetic training data that mimics text embeddings
        batch_size = 32
        num_batches = 200
        synthetic_data = []
        
        print("Generating realistic synthetic training data...")
        
        def create_realistic_embeddings(batch_size, embed_dim=768, num_classes=10):
            """Create synthetic embeddings that mimic realistic text feature distributions."""
            # Create class-specific prototype vectors
            class_prototypes = []
            for class_id in range(num_classes):
                # Each class has a different "semantic center"
                prototype = torch.randn(embed_dim) * 0.5
                
                # Add some structure to make classes distinguishable
                if class_id < 5:  # "Positive sentiment" classes
                    prototype[0:100] += 1.0  # Boost certain feature dimensions
                    prototype[200:300] += 0.5
                else:  # "Negative sentiment" classes  
                    prototype[0:100] -= 1.0  # Suppress those same dimensions
                    prototype[400:500] += 0.8
                
                # Add topic-specific features
                topic_start = (class_id % 3) * 200
                prototype[topic_start:topic_start+50] += 1.5
                
                class_prototypes.append(prototype)
            
            # Generate batch data
            inputs = []
            targets = []
            
            for _ in range(batch_size):
                # Randomly select a class
                class_id = torch.randint(0, num_classes, (1,)).item()
                target = class_id
                
                # Start with class prototype
                embedding = class_prototypes[class_id].clone()
                
                # Add realistic noise and variations
                # 1. Gaussian noise (like real embeddings have)
                embedding += torch.randn(embed_dim) * 0.3
                
                # 2. Add some sparsity (many dimensions near zero)
                mask = torch.rand(embed_dim) < 0.3  # 30% of dimensions get zeroed
                embedding[mask] = 0
                
                # 3. Add some extreme values (like attention peaks)
                extreme_indices = torch.randperm(embed_dim)[:20]
                embedding[extreme_indices] += torch.randn(20) * 2.0
                
                # 4. Normalize to realistic magnitude (like BERT embeddings)
                embedding = torch.nn.functional.normalize(embedding, dim=0) * 10.0
                
                inputs.append(embedding)
                targets.append(target)
            
            return torch.stack(inputs), torch.tensor(targets)
        
        # Generate multiple batches with realistic patterns
        for batch_idx in range(num_batches):
            inputs, targets = create_realistic_embeddings(batch_size)
            
            # Add some batch-specific characteristics
            if batch_idx % 2 == 0:
                # Even batches: simulate "formal text" - more structured
                inputs[:, 600:650] += 0.5  # Boost "formal language" features
            else:
                # Odd batches: simulate "informal text" - more varied
                inputs += torch.randn_like(inputs) * 0.2  # More noise
                inputs[:, 650:700] += 0.3  # Boost "informal language" features
            
            synthetic_data.append((inputs, targets))
            if (batch_idx + 1) % 50 == 0:
                print(f"  Generated batch {batch_idx + 1}: {inputs.shape}, class distribution: {torch.bincount(targets).tolist()}")
        
        # Create a simple DataLoader-like structure
        class SyntheticDataLoader:
            def __init__(self, data):
                self.data = data
            
            def __iter__(self):
                return iter(self.data)
            
            def __len__(self):
                return len(self.data)
        
        dataloader = SyntheticDataLoader(synthetic_data)
        
        # STEP 1: Train the teacher model first
        print("\nStep 1: Training teacher model...")
        teacher_optimizer = torch.optim.Adam(teacher_model.parameters(), lr=0.001)
        teacher_criterion = nn.CrossEntropyLoss()
        
        teacher_model.train()
        teacher_losses = []
        
        for epoch in range(20):  # Train teacher for more epochs
            epoch_loss = 0
            correct_predictions = 0
            total_predictions = 0
            
            for batch_inputs, batch_targets in dataloader:
                teacher_optimizer.zero_grad()
                
                # Forward pass
                outputs = teacher_model(batch_inputs)
                loss = teacher_criterion(outputs, batch_targets)
                
                # Backward pass
                loss.backward()
                teacher_optimizer.step()
                
                # Track metrics
                epoch_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total_predictions += batch_targets.size(0)
                correct_predictions += (predicted == batch_targets).sum().item()
            
            avg_loss = epoch_loss / len(dataloader)
            accuracy = 100 * correct_predictions / total_predictions
            teacher_losses.append(avg_loss)
            
            print(f"  Teacher Epoch {epoch + 1}/5 - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")
        
        # Evaluate teacher on test data
        print("\nEvaluating trained teacher model...")
        test_inputs, test_targets = create_realistic_embeddings(64)  # Larger test set
        
        teacher_model.eval()
        with torch.no_grad():
            teacher_test_outputs = teacher_model(test_inputs)
            teacher_test_loss = teacher_criterion(teacher_test_outputs, test_targets)
            _, teacher_test_preds = torch.max(teacher_test_outputs, 1)
            teacher_test_accuracy = (teacher_test_preds == test_targets).float().mean().item()
            
            print(f"  Teacher test accuracy: {teacher_test_accuracy:.2%}")
            print(f"  Teacher test loss: {teacher_test_loss:.4f}")
        
        # STEP 2: Now distill knowledge into student model
        print("\nStep 2: Distilling knowledge into student model...")
        distiller = ModelDistiller(teacher_model, student_model, temperature=4.0)
        student_optimizer = torch.optim.Adam(student_model.parameters(), lr=0.001)
        
        # Train student using distillation
        distiller.train_student(dataloader, student_optimizer, epochs=20)
        
        # STEP 3: Compare all models
        print("\nStep 3: Comparing model performance...")
        
        # Also train a baseline student without distillation for comparison
        print("Training baseline student (without distillation)...")
        baseline_student = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
        
        baseline_optimizer = torch.optim.Adam(baseline_student.parameters(), lr=0.001)
        baseline_criterion = nn.CrossEntropyLoss()
        
        baseline_student.train()
        for epoch in range(20):
            epoch_loss = 0
            for batch_inputs, batch_targets in dataloader:
                baseline_optimizer.zero_grad()
                outputs = baseline_student(batch_inputs)
                loss = baseline_criterion(outputs, batch_targets)
                loss.backward()
                baseline_optimizer.step()
                epoch_loss += loss.item()
            print(f"  Baseline Student Epoch {epoch + 1}/3 - Loss: {epoch_loss / len(dataloader):.4f}")
        
        # Final evaluation on test set
        teacher_model.eval()
        student_model.eval()
        baseline_student.eval()
        
        with torch.no_grad():
            # Measure inference times
            start_time = time.time()
            teacher_output = teacher_model(test_inputs)
            teacher_time = time.time() - start_time
            
            start_time = time.time()
            student_output = student_model(test_inputs)
            student_time = time.time() - start_time
            
            start_time = time.time()
            baseline_output = baseline_student(test_inputs)
            baseline_time = time.time() - start_time
            
            # Calculate accuracies
            _, teacher_preds = torch.max(teacher_output, 1)
            _, student_preds = torch.max(student_output, 1)
            _, baseline_preds = torch.max(baseline_output, 1)
            
            teacher_accuracy = (teacher_preds == test_targets).float().mean().item()
            student_accuracy = (student_preds == test_targets).float().mean().item()
            baseline_accuracy = (baseline_preds == test_targets).float().mean().item()
            
            # Calculate agreement between models
            teacher_student_agreement = (teacher_preds == student_preds).float().mean().item()
            teacher_baseline_agreement = (teacher_preds == baseline_preds).float().mean().item()
            
            print(f"\nFinal Results:")
            print(f"Model Performance:")
            print(f"  Teacher accuracy:           {teacher_accuracy:.2%}")
            print(f"  Distilled student accuracy: {student_accuracy:.2%}")
            print(f"  Baseline student accuracy:  {baseline_accuracy:.2%}")
            print(f"  Knowledge transfer gain:    {(student_accuracy - baseline_accuracy):.2%}")
            
            print(f"\nInference Speed:")
            print(f"  Teacher time:    {teacher_time:.4f}s")
            print(f"  Student time:    {student_time:.4f}s")
            print(f"  Baseline time:   {baseline_time:.4f}s")
            print(f"  Speedup:         {teacher_time / student_time:.2f}x")
            
            print(f"\nModel Agreement:")
            print(f"  Teacher-Student agreement:  {teacher_student_agreement:.2%}")
            print(f"  Teacher-Baseline agreement: {teacher_baseline_agreement:.2%}")
            
            # Show some example predictions
            print(f"\nSample predictions (first 8):")
            print(f"{'True':<6} {'Teacher':<8} {'Student':<8} {'Baseline':<8}")
            print("-" * 32)
            for i in range(min(8, len(test_targets))):
                print(f"{test_targets[i].item():<6} {teacher_preds[i].item():<8} {student_preds[i].item():<8} {baseline_preds[i].item():<8}")
            
    except Exception as e:
        print(f"Knowledge distillation test failed: {e}")
        import traceback
        traceback.print_exc()
    
    # 2. Quantization
    print("\n2. Testing quantization...")
    quantizer = ModelQuantizer(teacher_model)
    
    try:
        quantized_model = quantizer.dynamic_quantization()
        
        # Test performance with smaller input for faster testing
        test_input = torch.randn(8, 768)  # Reduced batch size
        print("Running performance comparison...")
        performance_stats = quantizer.compare_performance(test_input, iterations=50)
        
        print("Quantization Results:")
        for key, value in performance_stats.items():
            print(f"  {key}: {value}")
        
    except Exception as e:
        print(f"Quantization test failed: {e}")
    
    # 3. Inference Optimization
    print("\n3. Testing inference optimization...")
    try:
        optimizer = InferenceOptimizer(teacher_model)
        optimizer.optimize_model_for_inference()
        
        # Test batch inference
        test_texts = [f"Sample text {i}" for i in range(10)]
        print(f"Testing batch inference with {len(test_texts)} samples")
        # Note: This would work with actual tokenizer
        print("Batch inference optimization set up successfully")
        
    except Exception as e:
        print(f"Inference optimization test failed: {e}")
    
    # 4. Hardware Acceleration
    print("\n4. Testing hardware acceleration...")
    try:
        hw_manager = HardwareAccelerationManager()
        optimized_model, device = hw_manager.optimize_for_hardware(teacher_model)
        
        print(f"Model optimized for device: {device}")
        print("Hardware info:")
        for key, value in hw_manager.device_info.items():
            print(f"  {key}: {value}")
            
    except Exception as e:
        print(f"Hardware acceleration test failed: {e}")
    
    # 5. Caching Example
    print("\n5. Testing caching system...")
    try:
        cache = IntelligentCache(max_size=100, ttl=60)
        
        # Test cache operations
        cache.put("computation_1", {"result": "expensive_calculation", "value": 42})
        cache.put("computation_2", {"result": "another_calculation", "value": 84})
        
        result1 = cache.get("computation_1")
        result2 = cache.get("computation_2")
        result3 = cache.get("nonexistent_key")
        
        print(f"Cache test results:")
        print(f"  computation_1: {result1}")
        print(f"  computation_2: {result2}")
        print(f"  nonexistent_key: {result3}")
        
    except Exception as e:
        print(f"Caching test failed: {e}")
    
    # 6. Batch Processing Test
    print("\n6. Testing batch processing...")
    try:
        async def dummy_model_func(batch):
            # Simulate model processing
            await asyncio.sleep(0.1)  # Simulate processing time
            return [f"processed_{item}" for item in batch]
        
        batch_processor = BatchProcessor(dummy_model_func, batch_size=4, max_wait_time=0.2)
        print("Batch processor created successfully")
        print("(Note: Full async testing would require running in async context)")
        
    except Exception as e:
        print(f"Batch processing test failed: {e}")
    
    print("\n=== Model-Level Optimization Demo Complete ===")


In [49]:

# Run synchronous example
example_usage()

=== Model-Level Optimization Demo ===
PyTorch version: 2.7.1
Quantization available: True
Current quantization engine: qnnpack
Teacher model: 1,446,154 parameters
Student model: 231,050 parameters
Size reduction: 84.0%

1. Testing knowledge distillation...
Generating realistic synthetic training data...
  Generated batch 50: torch.Size([32, 768]), class distribution: [6, 1, 0, 0, 4, 5, 6, 4, 3, 3]
  Generated batch 100: torch.Size([32, 768]), class distribution: [3, 3, 3, 3, 2, 3, 4, 5, 1, 5]
  Generated batch 150: torch.Size([32, 768]), class distribution: [4, 2, 5, 3, 3, 1, 5, 5, 3, 1]
  Generated batch 200: torch.Size([32, 768]), class distribution: [3, 6, 1, 0, 2, 5, 3, 3, 6, 3]

Step 1: Training teacher model...
  Teacher Epoch 1/5 - Loss: 0.6894, Accuracy: 59.33%
  Teacher Epoch 2/5 - Loss: 0.5454, Accuracy: 66.23%
  Teacher Epoch 3/5 - Loss: 0.4444, Accuracy: 77.11%
  Teacher Epoch 4/5 - Loss: 0.3687, Accuracy: 83.39%
  Teacher Epoch 5/5 - Loss: 0.3158, Accuracy: 86.27%
  Teache

In [50]:
# Run async example
print("\nRunning async example...")
try:
    await async_example()
except Exception as e:
    print(f"Async example failed: {e}")
    print("This is normal if running in a Jupyter notebook - use await async_example() instead")


Running async example...

=== Async Features Demo ===
Starting async inference...
Would process 20 inputs asynchronously
Async processing completed in 0.50 seconds
=== Async Features Demo Complete ===


## Section 9.2 - System architecture and execution techniques

In [51]:
# Section 9.2: System Architecture and Execution Optimization

import asyncio
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional, Callable, Any, Tuple
import time
import heapq
from abc import ABC, abstractmethod
import queue
import psutil
import logging
from contextlib import asynccontextmanager

In [63]:
# ==========================================
# Task Priority and Resource Management
# ==========================================

class TaskPriority(Enum):
    CRITICAL = 1
    HIGH = 2
    MEDIUM = 3
    LOW = 4

@dataclass
class Task:
    id: str
    priority: TaskPriority
    function: Callable
    args: tuple = field(default_factory=tuple)
    kwargs: dict = field(default_factory=dict)
    estimated_duration: float = 1.0
    resource_requirements: dict = field(default_factory=dict)
    created_at: float = field(default_factory=time.time)
    
    def __lt__(self, other):
        return self.priority.value < other.priority.value

@dataclass 
class ResourceConstraints:
    max_cpu_percent: float = 80.0
    max_memory_mb: float = 4096.0
    max_concurrent_tasks: int = 10
    reserved_resources: dict = field(default_factory=dict)

# ==========================================
# Asynchronous Task Manager
# ==========================================

class AsyncTaskManager:
    """Manages asynchronous task execution with resource awareness."""
    
    def __init__(self, resource_constraints: ResourceConstraints):
        self.resource_constraints = resource_constraints
        self.task_queue = asyncio.PriorityQueue()
        self.active_tasks = {}
        self.completed_tasks = {}
        self.resource_monitor = ResourceMonitor()
        self.running = False
        
    async def submit_task(self, task: Task) -> str:
        """Submit a task for asynchronous execution."""
        await self.task_queue.put(task)
        return task.id
    
    async def start_processing(self):
        """Start the main processing loop."""
        self.running = True
        workers = []
        
        # Create worker coroutines
        for i in range(self.resource_constraints.max_concurrent_tasks):
            worker = asyncio.create_task(self._worker(f"worker-{i}"))
            workers.append(worker)
        
        # Start resource monitoring
        monitor_task = asyncio.create_task(self._monitor_resources())
        
        try:
            await asyncio.gather(*workers, monitor_task)
        except Exception as e:
            logging.error(f"Error in task processing: {e}")
        finally:
            self.running = False
    
    async def _worker(self, worker_id: str):
        """Worker coroutine that processes tasks from the queue."""
        while self.running:
            try:
                # Wait for task with timeout to allow graceful shutdown
                task = await asyncio.wait_for(self.task_queue.get(), timeout=1.0)
                
                # Check resource availability
                if not await self._can_execute_task(task):
                    # Put task back in queue and wait
                    await self.task_queue.put(task)
                    await asyncio.sleep(0.1)
                    continue
                
                # Execute task
                await self._execute_task(task, worker_id)
                
            except asyncio.TimeoutError:
                continue
            except Exception as e:
                logging.error(f"Worker {worker_id} error: {e}")
    
    async def _execute_task(self, task: Task, worker_id: str):
        """Execute a single task with monitoring."""
        start_time = time.time()
        self.active_tasks[task.id] = {
            'task': task,
            'worker_id': worker_id,
            'start_time': start_time
        }
        
        try:
            # Execute the task function
            if asyncio.iscoroutinefunction(task.function):
                result = await task.function(*task.args, **task.kwargs)
            else:
                # Run sync function in thread pool
                loop = asyncio.get_event_loop()
                result = await loop.run_in_executor(None, task.function, *task.args, **task.kwargs)
            
            # Store completion info
            self.completed_tasks[task.id] = {
                'result': result,
                'duration': time.time() - start_time,
                'worker_id': worker_id
            }
            
        except Exception as e:
            self.completed_tasks[task.id] = {
                'error': str(e),
                'duration': time.time() - start_time,
                'worker_id': worker_id
            }
        finally:
            # Remove from active tasks
            self.active_tasks.pop(task.id, None)
    
    async def _can_execute_task(self, task: Task) -> bool:
        """Check if resources are available to execute task."""
        current_resources = self.resource_monitor.get_current_usage()
        
        # Check CPU
        if current_resources['cpu_percent'] > self.resource_constraints.max_cpu_percent:
            return False
        
        # Check memory
        estimated_memory = task.resource_requirements.get('memory_mb', 100)
        if current_resources['memory_mb'] + estimated_memory > self.resource_constraints.max_memory_mb:
            return False
        
        return True
    
    async def _monitor_resources(self):
        """Monitor system resources continuously."""
        while self.running:
            self.resource_monitor.update_usage()
            await asyncio.sleep(1.0)

# ==========================================
# Resource Monitoring
# ==========================================

class ResourceMonitor:
    """Monitors system resource usage."""
    
    def __init__(self):
        self.current_usage = {
            'cpu_percent': 0.0,
            'memory_mb': 0.0,
            'disk_io': 0.0,
            'network_io': 0.0
        }
        self.history = []
        self.max_history = 100
    
    def update_usage(self):
        """Update current resource usage metrics."""
        self.current_usage = {
            'cpu_percent': psutil.cpu_percent(),
            'memory_mb': psutil.virtual_memory().used / (1024 * 1024),
            'disk_io': sum(psutil.disk_io_counters()[:2]) if psutil.disk_io_counters() else 0,
            'network_io': sum(psutil.net_io_counters()[:2]) if psutil.net_io_counters() else 0
        }
        
        # Add to history
        self.history.append({
            'timestamp': time.time(),
            **self.current_usage
        })
        
        # Limit history size
        if len(self.history) > self.max_history:
            self.history.pop(0)
    
    def get_current_usage(self) -> dict:
        """Get current resource usage."""
        return self.current_usage.copy()
    
    def get_average_usage(self, window_seconds: float = 60.0) -> dict:
        """Get average usage over time window."""
        cutoff_time = time.time() - window_seconds
        recent_history = [h for h in self.history if h['timestamp'] > cutoff_time]
        
        if not recent_history:
            return self.current_usage.copy()
        
        avg_usage = {}
        for key in self.current_usage.keys():
            avg_usage[key] = sum(h[key] for h in recent_history) / len(recent_history)
        
        return avg_usage

# ==========================================
# Parallelization Strategies
# ==========================================

class ParallelExecutor:
    """Manages different parallelization strategies."""
    
    def __init__(self, max_workers: int = None):
        self.max_workers = max_workers or min(32, (psutil.cpu_count() or 1) + 4)
    
    async def parallel_map(self, func: Callable, items: List[Any], batch_size: int = None) -> List[Any]:
        """Apply function to items in parallel."""
        if batch_size is None:
            batch_size = min(len(items), self.max_workers)
        
        # Split items into batches
        batches = [items[i:i + batch_size] for i in range(0, len(items), batch_size)]
        
        async def process_batch(batch):
            if asyncio.iscoroutinefunction(func):
                return await asyncio.gather(*[func(item) for item in batch])
            else:
                loop = asyncio.get_event_loop()
                with ThreadPoolExecutor(max_workers=len(batch)) as executor:
                    futures = [loop.run_in_executor(executor, func, item) for item in batch]
                    return await asyncio.gather(*futures)
        
        # Process batches in parallel
        batch_results = await asyncio.gather(*[process_batch(batch) for batch in batches])
        
        # Flatten results
        return [item for batch in batch_results for item in batch]
    
    def parallel_reduce(self, func: Callable, items: List[Any], initial_value: Any = None) -> Any:
        """Parallel reduce operation using divide-and-conquer."""
        if not items:
            return initial_value
        
        if len(items) == 1:
            return items[0] if initial_value is None else func(initial_value, items[0])
        
        # Divide
        mid = len(items) // 2
        left_half = items[:mid]
        right_half = items[mid:]
        
        with ThreadPoolExecutor(max_workers=2) as executor:
            left_future = executor.submit(self.parallel_reduce, func, left_half, initial_value)
            right_future = executor.submit(self.parallel_reduce, func, right_half, initial_value)
            
            left_result = left_future.result()
            right_result = right_future.result()
        
        # Conquer
        return func(left_result, right_result)

# ==========================================
# Predictive Execution System
# ==========================================

class PredictiveExecutor:
    """Implements predictive execution and speculative processing."""
    
    def __init__(self):
        self.prediction_cache = {}
        self.execution_history = []
        self.speculation_results = {}
    
    async def execute_with_prediction(self, task: Task, predict_next: Callable = None) -> Any:
        """Execute task and speculatively prepare for predicted next tasks."""
        # Execute current task
        current_result = await self._execute_task(task)
        
        # Record execution for learning
        self.execution_history.append({
            'task_id': task.id,
            'timestamp': time.time(),
            'duration': time.time() - task.created_at
        })
        
        # Predict and speculatively execute next likely tasks
        if predict_next:
            predicted_tasks = predict_next(task, current_result)
            await self._speculative_execution(predicted_tasks)
        
        return current_result
    
    async def _execute_task(self, task: Task) -> Any:
        """Execute a single task."""
        if asyncio.iscoroutinefunction(task.function):
            return await task.function(*task.args, **task.kwargs)
        else:
            loop = asyncio.get_event_loop()
            return await loop.run_in_executor(None, task.function, *task.args, **task.kwargs)
    
    async def _speculative_execution(self, predicted_tasks: List[Task]):
        """Execute predicted tasks speculatively in background."""
        for task in predicted_tasks:
            # Only speculate if not too expensive
            if task.estimated_duration < 5.0:  # seconds
                asyncio.create_task(self._speculative_task(task))
    
    async def _speculative_task(self, task: Task):
        """Execute speculative task and cache result."""
        try:
            result = await self._execute_task(task)
            self.speculation_results[task.id] = {
                'result': result,
                'timestamp': time.time(),
                'ttl': 300  # 5 minutes
            }
        except Exception as e:
            logging.warning(f"Speculative execution failed for task {task.id}: {e}")
    
    def get_speculative_result(self, task_id: str) -> Optional[Any]:
        """Retrieve speculative result if available and valid."""
        if task_id in self.speculation_results:
            spec_result = self.speculation_results[task_id]
            if time.time() - spec_result['timestamp'] < spec_result['ttl']:
                return spec_result['result']
            else:
                # Expired
                del self.speculation_results[task_id]
        return None

# ==========================================
# Connection Pooling and Warm Start
# ==========================================

class ConnectionPool:
    """Manages connection pooling for external services."""
    
    def __init__(self, max_connections: int = 10, idle_timeout: float = 300):
        self.max_connections = max_connections
        self.idle_timeout = idle_timeout
        self.pool = queue.Queue(maxsize=max_connections)
        self.active_connections = {}
        self.connection_factory = None
        self._lock = threading.Lock()
    
    def set_connection_factory(self, factory: Callable):
        """Set the factory function for creating new connections."""
        self.connection_factory = factory
    
    @asynccontextmanager
    async def get_connection(self):
        """Get a connection from the pool, creating if necessary."""
        connection = None
        try:
            # Try to get from pool
            connection = self._get_from_pool()
            if connection is None and self.connection_factory:
                connection = await self._create_connection()
            
            if connection:
                yield connection
            else:
                raise Exception("Unable to obtain connection")
                
        finally:
            if connection:
                self._return_to_pool(connection)
    
    def _get_from_pool(self):
        """Get connection from pool if available."""
        try:
            return self.pool.get_nowait()
        except queue.Empty:
            return None
    
    async def _create_connection(self):
        """Create new connection if under limit."""
        with self._lock:
            if len(self.active_connections) < self.max_connections:
                connection = await self.connection_factory()
                connection_id = id(connection)
                self.active_connections[connection_id] = {
                    'connection': connection,
                    'created_at': time.time(),
                    'last_used': time.time()
                }
                return connection
        return None
    
    def _return_to_pool(self, connection):
        """Return connection to pool."""
        connection_id = id(connection)
        if connection_id in self.active_connections:
            self.active_connections[connection_id]['last_used'] = time.time()
            try:
                self.pool.put_nowait(connection)
            except queue.Full:
                # Pool is full, close this connection
                self._close_connection(connection)
    
    def _close_connection(self, connection):
        """Close and remove connection."""
        connection_id = id(connection)
        if connection_id in self.active_connections:
            del self.active_connections[connection_id]
        # Implement actual connection closing based on connection type
        if hasattr(connection, 'close'):
            connection.close()
    
    def cleanup_idle_connections(self):
        """Remove idle connections that have exceeded timeout."""
        current_time = time.time()
        to_remove = []
        
        for conn_id, conn_info in self.active_connections.items():
            if current_time - conn_info['last_used'] > self.idle_timeout:
                to_remove.append(conn_info['connection'])
        
        for connection in to_remove:
            self._close_connection(connection)

# ==========================================
# Progressive Result Generation
# ==========================================

class ProgressiveResultGenerator:
    """Generates and streams results progressively."""
    
    def __init__(self):
        self.result_streams = {}
        self.partial_results = {}
    
    async def create_progressive_task(self, task_id: str, generator_func: Callable) -> asyncio.Queue:
        """Create a task that generates results progressively."""
        result_queue = asyncio.Queue()
        self.result_streams[task_id] = result_queue
        
        # Start the generator in background
        asyncio.create_task(self._run_progressive_generator(task_id, generator_func, result_queue))
        
        return result_queue
    
    async def _run_progressive_generator(self, task_id: str, generator_func: Callable, result_queue: asyncio.Queue):
        """Run the progressive generator and stream results."""
        try:
            # Call the generator function to get the generator object
            generator = generator_func()
            
            # Check if it's an async generator
            if hasattr(generator, '__aiter__'):
                # It's an async generator
                async for partial_result in generator:
                    await result_queue.put(('partial', partial_result))
                    self._update_partial_result(task_id, partial_result)
            else:
                # It's a sync generator
                for partial_result in generator:
                    await result_queue.put(('partial', partial_result))
                    self._update_partial_result(task_id, partial_result)
            
            # Signal completion
            final_result = self.partial_results.get(task_id)
            await result_queue.put(('complete', final_result))
            
        except Exception as e:
            await result_queue.put(('error', str(e)))
        finally:
            # Cleanup
            self.result_streams.pop(task_id, None)
            self.partial_results.pop(task_id, None)
    
    def _update_partial_result(self, task_id: str, partial_result):
        """Update the accumulated partial result."""
        if task_id not in self.partial_results:
            self.partial_results[task_id] = []
        self.partial_results[task_id].append(partial_result)
    
    async def get_progressive_results(self, task_id: str):
        """Generator to yield progressive results."""
        if task_id not in self.result_streams:
            return
        
        result_queue = self.result_streams[task_id]
        
        while True:
            try:
                result_type, result_data = await result_queue.get()
                
                if result_type == 'partial':
                    yield result_data
                elif result_type == 'complete':
                    yield result_data  # Yield the final result instead of returning it
                    return  # Exit without a value
                elif result_type == 'error':
                    raise Exception(result_data)
                    
            except asyncio.CancelledError:
                break

# ==========================================
# Critical Path Optimization
# ==========================================

class CriticalPathOptimizer:
    """Optimizes execution by identifying and prioritizing critical paths."""
    
    def __init__(self):
        self.task_dependencies = {}
        self.execution_times = {}
        self.critical_paths = {}
    
    def add_task_dependency(self, task_id: str, depends_on: List[str], estimated_time: float):
        """Add task dependency information."""
        self.task_dependencies[task_id] = depends_on
        self.execution_times[task_id] = estimated_time
    
    def calculate_critical_path(self, target_task: str) -> List[str]:
        """Calculate the critical path to a target task."""
        # Use dynamic programming to find longest path (critical path)
        memo = {}
        
        def longest_path(task_id: str) -> Tuple[float, List[str]]:
            if task_id in memo:
                return memo[task_id]
            
            dependencies = self.task_dependencies.get(task_id, [])
            if not dependencies:
                # Leaf task
                result = (self.execution_times.get(task_id, 0), [task_id])
            else:
                # Find the longest path among dependencies
                max_time = 0
                max_path = []
                
                for dep_task in dependencies:
                    dep_time, dep_path = longest_path(dep_task)
                    if dep_time > max_time:
                        max_time = dep_time
                        max_path = dep_path
                
                # Add current task
                total_time = max_time + self.execution_times.get(task_id, 0)
                full_path = max_path + [task_id]
                result = (total_time, full_path)
            
            memo[task_id] = result
            return result
        
        critical_time, critical_path = longest_path(target_task)
        self.critical_paths[target_task] = {
            'path': critical_path,
            'total_time': critical_time
        }
        
        return critical_path
    
    def get_priority_order(self, tasks: List[str]) -> List[str]:
        """Get execution order prioritizing critical path tasks."""
        critical_tasks = set()
        
        # Find all critical path tasks
        for task in tasks:
            if task not in self.critical_paths:
                self.calculate_critical_path(task)
            critical_tasks.update(self.critical_paths[task]['path'])
        
        # Sort tasks: critical path tasks first, then by estimated impact
        def task_priority(task_id: str):
            is_critical = 1 if task_id in critical_tasks else 2
            estimated_time = self.execution_times.get(task_id, 0)
            return (is_critical, -estimated_time)  # Negative for descending order
        
        return sorted(tasks, key=task_priority)

# ==========================================
# Cost-Aware Execution Planning
# ==========================================

class CostAwareExecutor:
    """Executes tasks with cost considerations and budget constraints."""
    
    def __init__(self, budget_limit: float, cost_model: Dict[str, float]):
        self.budget_limit = budget_limit
        self.cost_model = cost_model  # Cost per resource unit
        self.current_spend = 0.0
        self.execution_log = []
    
    def estimate_task_cost(self, task: Task) -> float:
        """Estimate the cost of executing a task."""
        base_cost = self.cost_model.get('base_execution', 0.01)
        cpu_cost = task.resource_requirements.get('cpu_units', 1) * self.cost_model.get('cpu_unit', 0.001)
        memory_cost = task.resource_requirements.get('memory_mb', 100) * self.cost_model.get('memory_mb', 0.0001)
        duration_cost = task.estimated_duration * self.cost_model.get('duration_second', 0.001)
        
        return base_cost + cpu_cost + memory_cost + duration_cost
    
    async def execute_within_budget(self, tasks: List[Task]) -> List[Tuple[str, Any]]:
        """Execute tasks while staying within budget constraints."""
        # Sort tasks by cost-effectiveness (priority vs cost)
        def cost_effectiveness(task: Task):
            cost = self.estimate_task_cost(task)
            priority_weight = 5 - task.priority.value  # Higher priority = lower number
            return priority_weight / max(cost, 0.001)  # Avoid division by zero
        
        sorted_tasks = sorted(tasks, key=cost_effectiveness, reverse=True)
        
        results = []
        for task in sorted_tasks:
            estimated_cost = self.estimate_task_cost(task)
            
            # Check if we can afford this task
            if self.current_spend + estimated_cost <= self.budget_limit:
                start_time = time.time()
                
                try:
                    # Execute task
                    if asyncio.iscoroutinefunction(task.function):
                        result = await task.function(*task.args, **task.kwargs)
                    else:
                        loop = asyncio.get_event_loop()
                        result = await loop.run_in_executor(None, task.function, *task.args, **task.kwargs)
                    
                    # Calculate actual cost based on execution time
                    actual_duration = time.time() - start_time
                    actual_cost = self._calculate_actual_cost(task, actual_duration)
                    
                    # Update spend and log
                    self.current_spend += actual_cost
                    self.execution_log.append({
                        'task_id': task.id,
                        'estimated_cost': estimated_cost,
                        'actual_cost': actual_cost,
                        'duration': actual_duration,
                        'timestamp': time.time()
                    })
                    
                    results.append((task.id, result))
                    
                except Exception as e:
                    # Still charge for failed execution
                    actual_duration = time.time() - start_time
                    actual_cost = self._calculate_actual_cost(task, actual_duration)
                    self.current_spend += actual_cost
                    
                    results.append((task.id, f"Error: {str(e)}"))
            else:
                # Skip task due to budget constraints
                results.append((task.id, "Skipped: Budget exceeded"))
        
        return results
    
    def _calculate_actual_cost(self, task: Task, actual_duration: float) -> float:
        """Calculate actual cost based on real execution metrics."""
        base_cost = self.cost_model.get('base_execution', 0.01)
        cpu_cost = task.resource_requirements.get('cpu_units', 1) * self.cost_model.get('cpu_unit', 0.001)
        memory_cost = task.resource_requirements.get('memory_mb', 100) * self.cost_model.get('memory_mb', 0.0001)
        duration_cost = actual_duration * self.cost_model.get('duration_second', 0.001)
        
        return base_cost + cpu_cost + memory_cost + duration_cost
    
    def get_budget_status(self) -> Dict[str, float]:
        """Get current budget status."""
        return {
            'budget_limit': self.budget_limit,
            'current_spend': self.current_spend,
            'remaining_budget': self.budget_limit - self.current_spend,
            'utilization_percent': (self.current_spend / self.budget_limit) * 100
        }

# ==========================================
# Example Usage and Testing
# ==========================================

async def example_system_optimization():
    """Demonstrate system architecture and execution optimization."""
    
    # Setup resource constraints
    constraints = ResourceConstraints(
        max_cpu_percent=75.0,
        max_memory_mb=2048.0,
        max_concurrent_tasks=5
    )
    
    # Create task manager
    task_manager = AsyncTaskManager(constraints)
    
    # Example tasks
    def cpu_intensive_task(duration):
        time.sleep(duration)
        return f"Completed CPU task in {duration}s"
    
    async def io_intensive_task(delay):
        await asyncio.sleep(delay)
        return f"Completed IO task in {delay}s"
    
    # Submit tasks with different priorities
    tasks = [
        Task("critical-1", TaskPriority.CRITICAL, cpu_intensive_task, (2,)),
        Task("high-1", TaskPriority.HIGH, io_intensive_task, (1,)),
        Task("medium-1", TaskPriority.MEDIUM, cpu_intensive_task, (1,)),
        Task("low-1", TaskPriority.LOW, io_intensive_task, (3,))
    ]
    
    # Submit tasks
    for task in tasks:
        await task_manager.submit_task(task)
    
    # Start processing (in real scenario, this would run continuously)
    print("Starting task processing...")
    
    # Example of parallel execution
    parallel_executor = ParallelExecutor(max_workers=4)
    
    # Example data to process in parallel
    data = list(range(1, 21))
    
    def square(x):
        return x * x
    
    # Parallel map operation
    squared_results = await parallel_executor.parallel_map(square, data, batch_size=5)
    print(f"Parallel map results: {squared_results}")
    
    # Parallel reduce operation
    sum_result = parallel_executor.parallel_reduce(lambda a, b: a + b, squared_results, 0)
    print(f"Parallel reduce result: {sum_result}")
    
    # Example of progressive result generation
    progressive_gen = ProgressiveResultGenerator()
    
    async def slow_computation():
        """Simulate a computation that yields intermediate results."""
        for i in range(5):
            await asyncio.sleep(0.5)
            yield f"Step {i+1} completed"
    
    # Create progressive task
    result_queue = await progressive_gen.create_progressive_task("prog-1", slow_computation)
    
    # Consume progressive results
    print("Progressive results:")
    async for result in progressive_gen.get_progressive_results("prog-1"):
        print(f"  {result}")
    
    # Example of cost-aware execution
    cost_model = {
        'base_execution': 0.01,
        'cpu_unit': 0.001,
        'memory_mb': 0.0001,
        'duration_second': 0.002
    }
    
    cost_executor = CostAwareExecutor(budget_limit=1.0, cost_model=cost_model)
    
    # Cost-aware task execution
    budget_tasks = [
        Task("budget-1", TaskPriority.HIGH, lambda: "High priority result", 
             resource_requirements={'cpu_units': 2, 'memory_mb': 512}, estimated_duration=1.0),
        Task("budget-2", TaskPriority.LOW, lambda: "Low priority result",
             resource_requirements={'cpu_units': 1, 'memory_mb': 256}, estimated_duration=2.0)
    ]
    
    budget_results = await cost_executor.execute_within_budget(budget_tasks)
    print(f"Budget execution results: {budget_results}")
    print(f"Budget status: {cost_executor.get_budget_status()}")

In [64]:
await example_system_optimization()

Starting task processing...
Parallel map results: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324, 361, 400]
Parallel reduce result: 2870
Progressive results:
  Step 1 completed
  Step 2 completed
  Step 3 completed
  Step 4 completed
  Step 5 completed
  ['Step 1 completed', 'Step 2 completed', 'Step 3 completed', 'Step 4 completed', 'Step 5 completed']
Budget execution results: [('budget-1', 'High priority result'), ('budget-2', 'Low priority result')]
Budget status: {'budget_limit': 1.0, 'current_spend': 0.09980234365463259, 'remaining_budget': 0.9001976563453674, 'utilization_percent': 9.980234365463259}


## Section 9.3:  Scalable Deployment and Infrastructure

In [ ]:
def deregister_service(self, service_name: str, instance_id: str):
        """Deregister a service instance."""
        instances = self.service_instances[service_name]
        self.service_instances[service_name] = [i for i in instances if i.id != instance_id]
        self.health_status.pop(f"{service_name}:{instance_id}", None)
        
        logging.info(f"Deregistered service instance: {service_name}:{instance_id}")
    
    def discover_service(self, service_name: str) -> List[ServerNode]:
        """Discover healthy instances of a service."""
        instances = self.service_instances.get(service_name, [])
        healthy_instances = []
        
        for instance in instances:
            health_key = f"{service_name}:{instance.id}"
            if self.health_status.get(health_key, False) and instance.is_healthy:
                healthy_instances.append(instance)
        
        return healthy_instances
    
    def get_service_instance(self, service_name: str, strategy: LoadBalancingStrategy = LoadBalancingStrategy.ROUND_ROBIN) -> Optional[ServerNode]:
        """Get a service instance using specified load balancing strategy."""
        instances = self.discover_service(service_name)
        if not instances:
            return None
        
        # Simple round-robin for service discovery
        if not hasattr(self, '_service_indices'):
            self._service_indices = {}
        
        if service_name not in self._service_indices:
            self._service_indices[service_name] = 0
        
        instance = instances[self._service_indices[service_name] % len(instances)]
        self._service_indices[service_name] += 1
        
        return instance

class MicroserviceManager:
    """Manages microservice lifecycle and communication."""
    
    def __init__(self):
        self.service_registry = ServiceRegistry()
        self.load_balancer = LoadBalancer()
        self.circuit_breakers = {}
        
    def register_microservice(self, service_name: str, instance: ServerNode):
        """Register a microservice instance."""
        self.service_registry.register_service(service_name, instance)
        self.load_balancer.add_server(instance)
        
        # Initialize circuit breaker
        self.circuit_breakers[f"{service_name}:{instance.id}"] = CircuitBreaker(
            failure_threshold=5,
            recovery_timeout=30
        )
    
    async def call_service(self, service_name: str, endpoint: str, data: dict = None, timeout: float = 10.0):
        """Make a call to a microservice with circuit breaker protection."""
        instance = self.service_registry.get_service_instance(service_name)
        if not instance:
            raise Exception(f"No healthy instances available for service: {service_name}")
        
        circuit_breaker_key = f"{service_name}:{instance.id}"
        circuit_breaker = self.circuit_breakers.get(circuit_breaker_key)
        
        if circuit_breaker and circuit_breaker.is_open():
            raise Exception(f"Circuit breaker is OPEN for {service_name}:{instance.id}")
        
        try:
            url = f"http://{instance.host}:{instance.port}{endpoint}"
            
            async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=timeout)) as session:
                if data:
                    async with session.post(url, json=data) as response:
                        result = await response.json()
                else:
                    async with session.get(url) as response:
                        result = await response.json()
                
                # Record success
                if circuit_breaker:
                    circuit_breaker.record_success()
                
                return result
                
        except Exception as e:
            # Record failure
            if circuit_breaker:
                circuit_breaker.record_failure()
            
            logging.error(f"Service call failed for {service_name}: {e}")
            raise

# ==========================================
# Circuit Breaker Pattern
# ==========================================

class CircuitBreakerState(Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"

class CircuitBreaker:
    """Circuit breaker pattern implementation for fault tolerance."""
    
    def __init__(self, failure_threshold: int = 5, recovery_timeout: float = 60.0):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.failure_count = 0
        self.success_count = 0
        self.last_failure_time = 0
        self.state = CircuitBreakerState.CLOSED
        
    def is_open(self) -> bool:
        """Check if circuit breaker is in open state."""
        if self.state == CircuitBreakerState.OPEN:
            # Check if recovery timeout has passed
            if time.time() - self.last_failure_time > self.recovery_timeout:
                self.state = CircuitBreakerState.HALF_OPEN
                return False
            return True
        return False
    
    def record_success(self):
        """Record a successful operation."""
        self.success_count += 1
        
        if self.state == CircuitBreakerState.HALF_OPEN:
            # Recovery successful
            self.state = CircuitBreakerState.CLOSED
            self.failure_count = 0
    
    def record_failure(self):
        """Record a failed operation."""
        self.failure_count += 1
        self.last_failure_time = time.time()
        
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitBreakerState.OPEN

# ==========================================
# Serverless Deployment Model
# ==========================================

class ServerlessFunction:
    """Represents a serverless function deployment."""
    
    def __init__(self, name: str, handler: Callable, memory_mb: int = 128, timeout: float = 30.0):
        self.name = name
        self.handler = handler
        self.memory_mb = memory_mb
        self.timeout = timeout
        self.invocation_count = 0
        self.total_duration = 0.0
        self.cold_starts = 0
        self.warm_instances = []
        self.max_warm_instances = 3
        
    async def invoke(self, event: dict, context: dict = None) -> dict:
        """Invoke the serverless function."""
        start_time = time.time()
        is_cold_start = len(self.warm_instances) == 0
        
        if is_cold_start:
            self.cold_starts += 1
            # Simulate cold start delay
            await asyncio.sleep(0.1)
        
        try:
            # Execute function
            if asyncio.iscoroutinefunction(self.handler):
                result = await self.handler(event, context)
            else:
                result = self.handler(event, context)
            
            execution_time = time.time() - start_time
            
            # Update metrics
            self.invocation_count += 1
            self.total_duration += execution_time
            
            # Manage warm instances
            if len(self.warm_instances) < self.max_warm_instances:
                self.warm_instances.append(time.time())
            
            return {
                'statusCode': 200,
                'body': result,
                'executionTime': execution_time,
                'coldStart': is_cold_start
            }
            
        except Exception as e:
            return {
                'statusCode': 500,
                'body': {'error': str(e)},
                'executionTime': time.time() - start_time,
                'coldStart': is_cold_start
            }
    
    def get_metrics(self) -> dict:
        """Get function performance metrics."""
        return {
            'invocation_count': self.invocation_count,
            'average_duration': self.total_duration / max(self.invocation_count, 1),
            'cold_start_rate': self.cold_starts / max(self.invocation_count, 1),
            'warm_instances': len(self.warm_instances)
        }

class ServerlessRuntime:
    """Serverless function runtime and orchestrator."""
    
    def __init__(self):
        self.functions = {}
        self.execution_queue = asyncio.Queue()
        self.metrics = defaultdict(dict)
        
    def deploy_function(self, function: ServerlessFunction):
        """Deploy a serverless function."""
        self.functions[function.name] = function
        logging.info(f"Deployed serverless function: {function.name}")
    
    async def invoke_function(self, function_name: str, event: dict, context: dict = None) -> dict:
        """Invoke a serverless function."""
        if function_name not in self.functions:
            return {
                'statusCode': 404,
                'body': {'error': f'Function {function_name} not found'}
            }
        
        function = self.functions[function_name]
        result = await function.invoke(event, context)
        
        # Update runtime metrics
        self.metrics[function_name] = function.get_metrics()
        
        return result
    
    async def auto_scale_functions(self):
        """Auto-scale functions based on usage patterns."""
        for function_name, function in self.functions.items():
            metrics = function.get_metrics()
            
            # Simple scaling logic based on invocation rate
            if metrics['invocation_count'] > 0:
                avg_duration = metrics['average_duration']
                
                # Adjust warm instance count based on usage
                if avg_duration > 1.0 and function.max_warm_instances < 10:
                    function.max_warm_instances += 1
                    logging.info(f"Scaled up warm instances for {function_name}")
                elif avg_duration < 0.1 and function.max_warm_instances > 1:
                    function.max_warm_instances -= 1
                    logging.info(f"Scaled down warm instances for {function_name}")

# ==========================================
# Performance Monitoring and Benchmarking
# ==========================================

class PerformanceMonitor:
    """Comprehensive performance monitoring system."""
    
    def __init__(self):
        self.metrics = defaultdict(list)
        self.alerts = []
        self.thresholds = {
            'response_time': 1.0,  # seconds
            'error_rate': 0.05,    # 5%
            'cpu_usage': 80.0,     # percent
            'memory_usage': 85.0   # percent
        }
        
    def record_request(self, service_name: str, endpoint: str, response_time: float, success: bool):
        """Record a request metric."""
        timestamp = time.time()
        self.metrics[f"{service_name}:{endpoint}"].append({
            'timestamp': timestamp,
            'response_time': response_time,
            'success': success
        })
        
        # Check for alerts
        self._check_alerts(service_name, endpoint, response_time, success)
    
    def record_system_metrics(self, service_name: str, cpu_usage: float, memory_usage: float):
        """Record system-level metrics."""
        timestamp = time.time()
        self.metrics[f"{service_name}:system"].append({
            'timestamp': timestamp,
            'cpu_usage': cpu_usage,
            'memory_usage': memory_usage
        })
        
        # Check system alerts
        if cpu_usage > self.thresholds['cpu_usage']:
            self._create_alert(f"High CPU usage for {service_name}: {cpu_usage:.1f}%")
        
        if memory_usage > self.thresholds['memory_usage']:
            self._create_alert(f"High memory usage for {service_name}: {memory_usage:.1f}%")
    
    def get_service_metrics(self, service_name: str, window_minutes: int = 10) -> dict:
        """Get aggregated metrics for a service."""
        cutoff_time = time.time() - (window_minutes * 60)
        service_metrics = {}
        
        for key, metrics_list in self.metrics.items():
            if key.startswith(service_name):
                recent_metrics = [m for m in metrics_list if m['timestamp'] > cutoff_time]
                
                if recent_metrics:
                    if 'response_time' in recent_metrics[0]:
                        # Request metrics
                        response_times = [m['response_time'] for m in recent_metrics]
                        successes = [m['success'] for m in recent_metrics]
                        
                        service_metrics[key] = {
                            'request_count': len(recent_metrics),
                            'avg_response_time': sum(response_times) / len(response_times),
                            'p95_response_time': np.percentile(response_times, 95),
                            'error_rate': 1 - (sum(successes) / len(successes)),
                            'throughput': len(recent_metrics) / window_minutes  # requests per minute
                        }
                    else:
                        # System metrics
                        cpu_values = [m['cpu_usage'] for m in recent_metrics if 'cpu_usage' in m]
                        memory_values = [m['memory_usage'] for m in recent_metrics if 'memory_usage' in m]
                        
                        if cpu_values:
                            service_metrics[key] = {
                                'avg_cpu_usage': sum(cpu_values) / len(cpu_values),
                                'max_cpu_usage': max(cpu_values),
                                'avg_memory_usage': sum(memory_values) / len(memory_values),
                                'max_memory_usage': max(memory_values)
                            }
        
        return service_metrics
    
    def _check_alerts(self, service_name: str, endpoint: str, response_time: float, success: bool):
        """Check if metrics breach alert thresholds."""
        if response_time > self.thresholds['response_time']:
            self._create_alert(f"Slow response for {service_name}{endpoint}: {response_time:.2f}s")
        
        if not success:
            # Calculate recent error rate
            key = f"{service_name}:{endpoint}"
            recent_requests = [m for m in self.metrics[key][-20:]]  # Last 20 requests
            if len(recent_requests) >= 5:
                error_rate = 1 - (sum(r['success'] for r in recent_requests) / len(recent_requests))
                if error_rate > self.thresholds['error_rate']:
                    self._create_alert(f"High error rate for {service_name}{endpoint}: {error_rate:.1%}")
    
    def _create_alert(self, message: str):
        """Create an alert."""
        alert = {
            'timestamp': time.time(),
            'message': message,
            'severity': 'warning'
        }
        self.alerts.append(alert)
        logging.warning(f"ALERT: {message}")
        
        # Keep only recent alerts
        cutoff_time = time.time() - 3600  # 1 hour
        self.alerts = [a for a in self.alerts if a['timestamp'] > cutoff_time]

class BenchmarkSuite:
    """Comprehensive benchmarking for agentic systems."""
    
    def __init__(self):
        self.benchmark_results = {}
        
    async def run_load_test(self, target_func: Callable, concurrent_requests: int = 10, 
                           duration_seconds: int = 60) -> dict:
        """Run a load test against a target function."""
        start_time = time.time()
        end_time = start_time + duration_seconds
        
        results = []
        semaphore = asyncio.Semaphore(concurrent_requests)
        
        async def single_request():
            async with semaphore:
                request_start = time.time()
                try:
                    await target_func()
                    success = True
                    error = None
                except Exception as e:
                    success = False
                    error = str(e)
                
                return {
                    'timestamp': request_start,
                    'response_time': time.time() - request_start,
                    'success': success,
                    'error': error
                }
        
        # Generate load
        tasks = []
        while time.time() < end_time:
            task = asyncio.create_task(single_request())
            tasks.append(task)
            await asyncio.sleep(1.0 / concurrent_requests)  # Rate limiting
        
        # Wait for all requests to complete
        results = await asyncio.gather(*tasks, return_exceptions=True)
        results = [r for r in results if isinstance(r, dict)]
        
        # Calculate metrics
        if results:
            response_times = [r['response_time'] for r in results]
            successes = [r['success'] for r in results]
            
            return {
                'total_requests': len(results),
                'successful_requests': sum(successes),
                'error_rate': 1 - (sum(successes) / len(successes)),
                'avg_response_time': sum(response_times) / len(response_times),
                'min_response_time': min(response_times),
                'max_response_time': max(response_times),
                'p50_response_time': np.percentile(response_times, 50),
                'p95_response_time': np.percentile(response_times, 95),
                'p99_response_time': np.percentile(response_times, 99),
                'throughput': len(results) / duration_seconds
            }
        
        return {'error': 'No successful requests completed'}
    
    async def benchmark_agent_operations(self, agent_func: Callable, test_cases: List[dict]) -> dict:
        """Benchmark specific agent operations with different inputs."""
        operation_results = {}
        
        for i, test_case in enumerate(test_cases):
            case_name = test_case.get('name', f'case_{i}')
            inputs = test_case.get('inputs', {})
            
            # Run multiple iterations
            iterations = test_case.get('iterations', 10)
            case_results = []
            
            for _ in range(iterations):
                start_time = time.time()
                try:
                    result = await agent_func(**inputs)
                    execution_time = time.time() - start_time
                    case_results.append({
                        'execution_time': execution_time,
                        'success': True,
                        'result_size': len(str(result)) if result else 0
                    })
                except Exception as e:
                    execution_time = time.time() - start_time
                    case_results.append({
                        'execution_time': execution_time,
                        'success':# Section 9.3: Scalable Deployment and Infrastructure

import asyncio
import aiohttp
import json
import time
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Callable, Any, Tuple
from abc import ABC, abstractmethod
from enum import Enum
import logging
import hashlib
import psutil
from collections import defaultdict, deque
import threading
import weakref

# ==========================================
# Load Balancing Implementation
# ==========================================

class LoadBalancingStrategy(Enum):
    ROUND_ROBIN = "round_robin"
    LEAST_CONNECTIONS = "least_connections"
    WEIGHTED_ROUND_ROBIN = "weighted_round_robin"
    LEAST_RESPONSE_TIME = "least_response_time"
    RESOURCE_BASED = "resource_based"

@dataclass
class ServerNode:
    id: str
    host: str
    port: int
    weight: float = 1.0
    current_connections: int = 0
    total_requests: int = 0
    avg_response_time: float = 0.0
    cpu_usage: float = 0.0
    memory_usage: float = 0.0
    is_healthy: bool = True
    last_health_check: float = field(default_factory=time.time)

class LoadBalancer:
    """Advanced load balancer with multiple strategies and health checking."""
    
    def __init__(self, strategy: LoadBalancingStrategy = LoadBalancingStrategy.ROUND_ROBIN):
        self.strategy = strategy
        self.servers = []
        self.current_index = 0
        self.request_history = defaultdict(deque)
        self.health_check_interval = 30  # seconds
        self.health_check_timeout = 5
        self._lock = threading.Lock()
        
    def add_server(self, server: ServerNode):
        """Add a server to the load balancer."""
        with self._lock:
            self.servers.append(server)
    
    def remove_server(self, server_id: str):
        """Remove a server from the load balancer."""
        with self._lock:
            self.servers = [s for s in self.servers if s.id != server_id]
    
    def get_next_server(self) -> Optional[ServerNode]:
        """Get the next server based on the load balancing strategy."""
        healthy_servers = [s for s in self.servers if s.is_healthy]
        
        if not healthy_servers:
            return None
        
        if self.strategy == LoadBalancingStrategy.ROUND_ROBIN:
            return self._round_robin(healthy_servers)
        elif self.strategy == LoadBalancingStrategy.LEAST_CONNECTIONS:
            return self._least_connections(healthy_servers)
        elif self.strategy == LoadBalancingStrategy.WEIGHTED_ROUND_ROBIN:
            return self._weighted_round_robin(healthy_servers)
        elif self.strategy == LoadBalancingStrategy.LEAST_RESPONSE_TIME:
            return self._least_response_time(healthy_servers)
        elif self.strategy == LoadBalancingStrategy.RESOURCE_BASED:
            return self._resource_based(healthy_servers)
        
        return healthy_servers[0]  # fallback
    
    def _round_robin(self, servers: List[ServerNode]) -> ServerNode:
        """Simple round-robin selection."""
        with self._lock:
            server = servers[self.current_index % len(servers)]
            self.current_index += 1
            return server
    
    def _least_connections(self, servers: List[ServerNode]) -> ServerNode:
        """Select server with least active connections."""
        return min(servers, key=lambda s: s.current_connections)
    
    def _weighted_round_robin(self, servers: List[ServerNode]) -> ServerNode:
        """Weighted round-robin based on server weights."""
        total_weight = sum(s.weight for s in servers)
        random_weight = random.uniform(0, total_weight)
        
        cumulative_weight = 0
        for server in servers:
            cumulative_weight += server.weight
            if random_weight <= cumulative_weight:
                return server
        
        return servers[-1]  # fallback
    
    def _least_response_time(self, servers: List[ServerNode]) -> ServerNode:
        """Select server with lowest average response time."""
        return min(servers, key=lambda s: s.avg_response_time)
    
    def _resource_based(self, servers: List[ServerNode]) -> ServerNode:
        """Select server based on resource utilization."""
        def resource_score(server):
            cpu_factor = 1 - (server.cpu_usage / 100)
            memory_factor = 1 - (server.memory_usage / 100)
            return (cpu_factor + memory_factor) / 2
        
        return max(servers, key=resource_score)
    
    async def health_check(self):
        """Perform health checks on all servers."""
        async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=self.health_check_timeout)) as session:
            tasks = []
            for server in self.servers:
                task = asyncio.create_task(self._check_server_health(session, server))
                tasks.append(task)
            
            await asyncio.gather(*tasks, return_exceptions=True)
    
    async def _check_server_health(self, session: aiohttp.ClientSession, server: ServerNode):
        """Check health of a single server."""
        try:
            url = f"http://{server.host}:{server.port}/health"
            start_time = time.time()
            
            async with session.get(url) as response:
                response_time = time.time() - start_time
                
                if response.status == 200:
                    server.is_healthy = True
                    # Update response time (exponential moving average)
                    alpha = 0.1
                    server.avg_response_time = (alpha * response_time + 
                                               (1 - alpha) * server.avg_response_time)
                else:
                    server.is_healthy = False
                    
        except Exception as e:
            logging.warning(f"Health check failed for server {server.id}: {e}")
            server.is_healthy = False
        
        server.last_health_check = time.time()
    
    def update_server_metrics(self, server_id: str, metrics: Dict[str, float]):
        """Update server metrics for load balancing decisions."""
        with self._lock:
            for server in self.servers:
                if server.id == server_id:
                    server.cpu_usage = metrics.get('cpu_usage', server.cpu_usage)
                    server.memory_usage = metrics.get('memory_usage', server.memory_usage)
                    server.current_connections = metrics.get('connections', server.current_connections)
                    break

# ==========================================
# Auto-Scaling Implementation
# ==========================================

class ScalingPolicy:
    """Defines when and how to scale resources."""
    
    def __init__(self, 
                 scale_up_threshold: float = 70.0,
                 scale_down_threshold: float = 30.0,
                 min_instances: int = 1,
                 max_instances: int = 10,
                 cooldown_period: int = 300):
        self.scale_up_threshold = scale_up_threshold
        self.scale_down_threshold = scale_down_threshold
        self.min_instances = min_instances
        self.max_instances = max_instances
        self.cooldown_period = cooldown_period
        self.last_scaling_action = 0

class AutoScaler:
    """Implements automatic horizontal scaling based on metrics."""
    
    def __init__(self, scaling_policy: ScalingPolicy):
        self.policy = scaling_policy
        self.instances = []
        self.metrics_history = deque(maxlen=100)
        self.instance_factory = None
        
    def set_instance_factory(self, factory: Callable[[], ServerNode]):
        """Set factory function for creating new instances."""
        self.instance_factory = factory
    
    def add_initial_instances(self, instances: List[ServerNode]):
        """Add initial set of instances."""
        self.instances.extend(instances)
    
    async def evaluate_scaling(self, current_metrics: Dict[str, float]) -> Dict[str, Any]:
        """Evaluate whether scaling action is needed."""
        self.metrics_history.append({
            'timestamp': time.time(),
            **current_metrics
        })
        
        # Check cooldown period
        if time.time() - self.policy.last_scaling_action < self.policy.cooldown_period:
            return {'action': 'none', 'reason': 'cooldown_period'}
        
        # Calculate average metrics over recent history
        if len(self.metrics_history) < 3:
            return {'action': 'none', 'reason': 'insufficient_history'}
        
        recent_metrics = list(self.metrics_history)[-5:]  # Last 5 measurements
        avg_cpu = sum(m.get('cpu_usage', 0) for m in recent_metrics) / len(recent_metrics)
        avg_memory = sum(m.get('memory_usage', 0) for m in recent_metrics) / len(recent_metrics)
        avg_load = max(avg_cpu, avg_memory)
        
        current_instance_count = len([i for i in self.instances if i.is_healthy])
        
        # Scaling decision logic
        if avg_load > self.policy.scale_up_threshold and current_instance_count < self.policy.max_instances:
            new_instance = await self._scale_up()
            return {
                'action': 'scale_up',
                'new_instance': new_instance.id if new_instance else None,
                'trigger_load': avg_load,
                'instance_count': len(self.instances)
            }
        
        elif avg_load < self.policy.scale_down_threshold and current_instance_count > self.policy.min_instances:
            removed_instance = await self._scale_down()
            return {
                'action': 'scale_down',
                'removed_instance': removed_instance.id if removed_instance else None,
                'trigger_load': avg_load,
                'instance_count': len(self.instances)
            }
        
        return {'action': 'none', 'reason': 'within_thresholds', 'current_load': avg_load}
    
    async def _scale_up(self) -> Optional[ServerNode]:
        """Add a new instance."""
        if not self.instance_factory:
            logging.error("No instance factory configured")
            return None
        
        try:
            new_instance = self.instance_factory()
            self.instances.append(new_instance)
            self.policy.last_scaling_action = time.time()
            
            logging.info(f"Scaled up: Added instance {new_instance.id}")
            return new_instance
            
        except Exception as e:
            logging.error(f"Failed to scale up: {e}")
            return None
    
    async def _scale_down(self) -> Optional[ServerNode]:
        """Remove an instance (least loaded)."""
        if not self.instances:
            return None
        
        # Find instance with least load to remove
        healthy_instances = [i for i in self.instances if i.is_healthy]
        if len(healthy_instances) <= self.policy.min_instances:
            return None
        
        # Select instance with lowest utilization
        instance_to_remove = min(healthy_instances, 
                               key=lambda i: max(i.cpu_usage, i.memory_usage))
        
        # Gracefully drain connections before removal
        await self._drain_instance(instance_to_remove)
        
        self.instances.remove(instance_to_remove)
        self.policy.last_scaling_action = time.time()
        
        logging.info(f"Scaled down: Removed instance {instance_to_remove.id}")
        return instance_to_remove
    
    async def _drain_instance(self, instance: ServerNode, timeout: float = 60.0):
        """Gracefully drain connections from an instance."""
        start_time = time.time()
        
        # Mark instance as unhealthy to stop new requests
        instance.is_healthy = False
        
        # Wait for existing connections to finish
        while instance.current_connections > 0 and (time.time() - start_time) < timeout:
            await asyncio.sleep(1.0)
        
        logging.info(f"Drained instance {instance.id} in {time.time() - start_time:.2f}s")

# ==========================================
# Microservices Architecture
# ==========================================

class ServiceRegistry:
    """Service discovery and registration for microservices."""
    
    def __init__(self):
        self.services = {}
        self.service_instances = defaultdict(list)
        self.health_status = {}
        
    def register_service(self, service_name: str, instance: ServerNode):
        """Register a service instance."""
        self.service_instances[service_name].append(instance)
        self.health_status[f"{service_name}:{instance.id}"] = True
        
        logging.info(f"Registered service instance: {service_name}:{instance.id}")
    
    def deregister_service(self, service_name: str, instance_id: str):
        """Deregister a service instance."""
        instances = self.service_instances[service_name]
        self.service_instances[service_name] = [i for i in instances if i.id !=